## Accessing Animal Tracking - Satellite Relay Tagging - Delayed Mode - SSM locations (Parquet)
This Jupyter notebook demonstrates how to access and plot animal_ssmoutputs_satellite_relay_tagging_location_qc_delayed_qc data, available as a [Parquet](https://parquet.apache.org) dataset stored on S3.

🔗 More information about the dataset is available [in the AODN metadata catalogue](https://catalogue-imos.aodn.org.au/geonetwork/srv/eng/catalog.search#/metadata/70f148b1-7040-4fad-944a-456413c95472).

📌 The source of truth for this notebook is maintained on [GitHub](https://github.com/aodn/aodn_cloud_optimised/tree/main/notebooks/animal_ssmoutputs_satellite_relay_tagging_location_qc_delayed_qc.ipynb).


In [ ]:
dataset_name = "animal_ssmoutputs_satellite_relay_tagging_location_qc_delayed_qc"

## Install/Update packages and Load common functions

In [ ]:
import os, requests, importlib.util

open('setup.py', 'w').write(requests.get('https://raw.githubusercontent.com/aodn/aodn_cloud_optimised/main/notebooks/setup.py').text)

spec = importlib.util.spec_from_file_location("setup", "setup.py")
setup = importlib.util.module_from_spec(spec)
spec.loader.exec_module(setup)

setup.install_requirements()
setup.load_dataquery()

In [ ]:
from DataQuery import GetAodn

# Understanding the Dataset

## Understanding Parquet Partitioning

Parquet files can be **partitioned** by one or more columns, which means the data is physically organised into folders based on the values in those columns. This is similar to how databases use indexes to optimise query performance.

Partitioning enables **faster filtering**: when you query data using a partitioned column, only the relevant subset of files needs to be read—improving performance significantly.

For example, if a dataset is partitioned by `"site_code"`, `"timestamp"`, and `"polygon"`, filtering on `"site_code"` allows the system to skip unrelated files entirely.

In this notebook, the `GetAodn` class includes built-in methods to efficiently filter data by **time** and **latitude/longitude** using the **timestamp** and **polygon** partitions. Other partitions can be used for filtering via the `scalar_filter`.

Any filtering on columns that are **not** partitioned can be significantly slower, as all files may need to be scanned. However, the `GetAodn` class provides a `scalar_filter` method that lets you apply these filters at load time—before the data is fully read—helping reduce the size of the resulting DataFrame.

Once the dataset is loaded, further filtering using Pandas is efficient and flexible.

See further below in the notebook for examples of how to filter the data effectively.

To view the actual partition columns for this dataset, run:


In [ ]:
aodn = GetAodn()
dname = f'{dataset_name}.parquet'
%time aodn_dataset = aodn.get_dataset(dname)

In [ ]:
aodn_dataset.dataset.partitioning.schema

## List unique partition values

In [ ]:
%%time
unique_partition_value = aodn_dataset.get_unique_partition_values('YOUR_PARTITION_KEY')
print(list(unique_partition_value)[0:2])  # showing a subset only

## Visualise Spatial Extent of the dataset
This section plots the polygons representing the areas where data is available. It helps to identify and create a bounding box around the regions containing data.

In [ ]:
aodn_dataset.plot_spatial_extent()

## Get Temporal Extent of the dataset

Similary to the spatial extent, we're retrieving the minimum and maximum timestamp partition values of the dataset. This is not necessarely accurately representative of the TIME values, as the timestamp partition can be yearly/monthly... but is here to give an idea

In [ ]:
%%time
aodn_dataset.get_temporal_extent()

## Read Metadata

For all Parquet datasets, we create a sidecar file named **_common_metadata** in the root of the dataset. This file contains both the dataset-level and variable-level attributes.  
The metadata can be retrieved below as a dictionary, and it will also be included in the pandas DataFrame when using the `get_data` method from the `GetAodn` class.

In [ ]:
metadata = aodn_dataset.get_metadata()
metadata

# Data Query and Plot

## Create a TIME and BoundingBox filter

This cell loads a subset of the dataset based on a time range and a spatial bounding box. The result is returned as a pandas DataFrame, and basic information about its structure is displayed.

In [ ]:
%%time
df = aodn_dataset.get_data(date_start='2019-12-01',
                           date_end='2020-10-15')

df.info()

In [ ]:
df

In [ ]:
## Download Subsetted Data as CSV

# This cell downloads the filtered dataset as a ZIP-compressed CSV file.  
# The CSV includes metadata at the top as commented lines, and a `FileLink` object is returned to allow downloading directly from the notebook.


#df.aodn.download_as_csv()

In [ ]:
import os
import matplotlib.animation as animation
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import HTML

# Increase memory limit for the animation
plt.rcParams["animation.embed_limit"] = 100.0

# =========================================================================
# 1. DATA PREPARATION (Mapping your column names)
# =========================================================================
# Ensure dates are datetime objects
df["date"] = pd.to_datetime(df["date"])
df = df.dropna(subset=["date", "lon", "lat"])

# Sort and get unique identifiers
df = df.sort_values(by="date")
unique_days = sorted(df["date"].dt.date.unique())
# 'ref' is the unique identifier for your animals
animal_ids = df["ref"].unique() 

# Chronological accumulation
track_history = {day: {aid: ([], []) for aid in animal_ids} for day in unique_days}
accumulated_data = {aid: ([], []) for aid in animal_ids}
day_groups = df.groupby(df["date"].dt.date)

for day in unique_days:
    if day in day_groups.groups:
        day_df = day_groups.get_group(day)
        for aid, animal_df in day_df.groupby("ref"):
            lon_list, lat_list = accumulated_data[aid]
            lon_list.extend(animal_df["lon"].tolist())
            lat_list.extend(animal_df["lat"].tolist())
    
    for aid in animal_ids:
        track_history[day][aid] = (
            list(accumulated_data[aid][0]),
            list(accumulated_data[aid][1]),
        )

# =========================================================================
# 2. ANIMATION SETUP
# =========================================================================
fig, ax = plt.subplots(figsize=(10, 8))

# Set map boundaries with padding
pad = 1.0
lon_min, lon_max = df["lon"].min() - pad, df["lon"].max() + pad
lat_min, lat_max = df["lat"].min() - pad, df["lat"].max() + pad

ax.set_xlim(lon_min, lon_max)
ax.set_ylim(lat_min, lat_max)
ax.grid(True, linestyle="--", alpha=0.5)

# Initialize lines for each animal
lines = {}
for aid in animal_ids:
    (line,) = ax.plot([], [], marker='.', markersize=4, linestyle="-", linewidth=1.5, label=aid)
    lines[aid] = line

title_text = ax.text(0.5, 1.02, "", transform=ax.transAxes, ha="center")
plt.legend(loc='upper right', fontsize='small')

def update(frame):
    current_day = unique_days[frame]
    for aid, line in lines.items():
        lons, lats = track_history[current_day][aid]
        line.set_data(lons, lats)
    title_text.set_text(f"Animal Tracks\nDate: {current_day}")
    return list(lines.values()) + [title_text]

ani = animation.FuncAnimation(
    fig, update, frames=len(unique_days), interval=200, blit=True
)
plt.close()

# Display the animation
HTML(ani.to_jshtml())